# Etapa 3 — Modelagem Preditiva: ARIMA / SARIMAX (`statsmodels`)

**Objetivo:** Modelar a série diária representativa **`MT_206`** (`1.461` dias, `freq='D'`) seguindo rigorosamente a sequência metodológica: visualização de tendência/sazonalidade (com decomposição STL), ACF/PACF, teste ADF e diferenciação até estacionariedade, separação cronológica treino/teste sem *data leakage*, ajuste e seleção por ACF/PACF e AIC/BIC, diagnóstico de resíduos (Ljung-Box e normalidade) e previsão fora da amostra com IC de 95% comparada contra *baselines*.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings("ignore", category=UserWarning)

RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "Codigo" else Path.cwd()
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from Codigo.src.arima_modelagem import (
    CANDIDATOS_MODELOS,
    CLIENTE_MODELAGEM,
    HORIZONTE_TESTE_DIAS,
    LAGS_LJUNG_BOX,
    NOME_MODELO_PRINCIPAL,
    PERIODO_SAZONAL_DIARIO,
    SEMENTE_PADRAO,
    avaliar_modelos_candidatos,
    executar_bateria_adf,
    gerar_figuras_arima,
    salvar_saidas_arima,
    separar_treino_teste_cronologico,
)
from Codigo.src.preparacao import carregar_serie_tratada

np.random.seed(SEMENTE_PADRAO)
print(f"Raiz do projeto: {RAIZ_PROJETO} | Cliente: {CLIENTE_MODELAGEM}")

## 1. Carga da Série e Separação Cronológica Treino/Teste

In [ ]:
df_diario = carregar_serie_tratada(
    RAIZ_PROJETO / "Dados/tratado/consumo_diario_clientes_completos_2011_2014.parquet", freq="D"
)
serie = df_diario[CLIENTE_MODELAGEM].copy()
treino, teste, info_split = separar_treino_teste_cronologico(serie, HORIZONTE_TESTE_DIAS)
print(json.dumps(info_split, indent=2, ensure_ascii=False))

## 2. Testes ADF e Diferenciação até Estacionariedade

In [ ]:
df_adf = executar_bateria_adf(treino, serie_completa=serie, s=PERIODO_SAZONAL_DIARIO)
display(df_adf)

## 3. Ajuste de Modelos Candidatos, Diagnóstico de Resíduos e Previsão vs. Baselines

In [ ]:
df_comparacao, detalhes_modelos = avaliar_modelos_candidatos(
    treino=treino, teste=teste, candidatos=CANDIDATOS_MODELOS, lags_lb=LAGS_LJUNG_BOX
)
display(df_comparacao)

figuras = gerar_figuras_arima(
    serie_completa=serie,
    treino=treino,
    teste=teste,
    df_adf=df_adf,
    detalhes_modelos=detalhes_modelos,
    nome_modelo_principal=NOME_MODELO_PRINCIPAL,
    diretorio_figuras=RAIZ_PROJETO / "Documentacao/figuras",
)
figuras_rel = [str(Path(f).relative_to(RAIZ_PROJETO)) for f in figuras]
salvar_saidas_arima(
    CLIENTE_MODELAGEM, info_split, df_adf, df_comparacao, detalhes_modelos,
    teste, figuras_rel, NOME_MODELO_PRINCIPAL, RAIZ_PROJETO / "Resultados",
)

for fig_path in figuras:
    display(Image(filename=fig_path))